In [1]:
import sys

if "google.colab" in sys.modules:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    original_data = '/content/drive/My Drive/MS1/original_dataset'
    final_data = '/content/drive/My Drive/MS1/Final_Dataset'

    # Install required packages
    !pip install pymatgen torch_geometric mp_api

else:
    original_data = "original_dataset"
    final_data = "Final_Dataset"

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, NNConv, CGConv, global_max_pool, GCNConv

from pymatgen.core import Structure, PeriodicSite, DummySpecie, Composition, Element
from pymatgen.core.periodic_table import Element as PMGElement
from pymatgen.analysis.local_env import MinimumDistanceNN, CrystalNN, VoronoiNN
# from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# import json
# import config
# import graphy_gnn
import graphy_cvae

/home/amutua/inverse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Prep

In [3]:
# The whole dataset
comb_df = pd.read_csv(f"{final_data}/combined/combined_data.csv")

# Sample row
sample_row = comb_df.iloc[0]

# Get defective structure
material = sample_row["dataset_material"]
id = sample_row["_id"]

# Get defective structure
defective_file_path = f"{original_data}/{material}/cifs/{id}.cif"
defective_structure = Structure.from_file(defective_file_path)

# Get reference structure
ref_file_path = f"{final_data}/ref_cifs/{material}.cif"
reference_structure = Structure.from_file(ref_file_path)

/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 32 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


## Work with Full Defective Structures

In [4]:
# Create a full defective structure instead of a cloud structure
def struct_to_dict(structure):
    rounded_coords = np.round(structure.frac_coords, 3)
    return {tuple(coord): site for coord, site in zip(rounded_coords, structure.sites)}


def get_defects_structure(reference_struct, defective_struct):
    # mindnn = MinimumDistanceNN()
    # struct to dict
    defective_dict = struct_to_dict(defective_struct)
    reference_dict = struct_to_dict(reference_struct)

    # Get lattice of defective structure
    structure_lattice = defective_struct.lattice

    # List to add all defect sites
    defective_structure_list = []

    # Dictionary to hold properties of each site
    defects_properties = {}

    ref_index = 0

    for ref_coord, ref_site in reference_dict.items():
        # Use the reference coordinates to get the defective site
        ref_index = ref_index + 1

        def_site = defective_dict.get(ref_coord)

        if def_site:  # The site is found in both the reference structure and the defective structure
            # But are the species the same?
            ref_specie = ref_site.specie
            def_specie = def_site.specie
            if ref_specie != def_specie:  # Substitution
                # Add site to defects list
                defective_structure_list.append(def_site)

                # Get atomic number change and defect type
                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": def_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": def_specie.X,

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": def_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": def_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": def_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(def_specie.common_oxidation_states) if def_specie.common_oxidation_states else 0,

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": def_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 1.0,
                    "normal_site":0.0,
                }

                defects_properties[def_site] = add_property
            else: # Normal site
                defective_structure_list.append(ref_site)

                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": ref_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": ref_specie.X,

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": ref_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": ref_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": ref_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(ref_specie.common_oxidation_states),

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": ref_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 0.0,
                    "normal_site":1.0
                }

                defects_properties[ref_site] = add_property

    
        else: # the site from ref_structure is not found in defective structure
            # This means that the site is a vacancy site
            # Add site to defective structure
            vacant_site = PeriodicSite(
                species= DummySpecie(),
                coords= ref_coord,
                coords_are_cartesian= False,
                lattice= structure_lattice
                )

            # Add site to defects list
            defective_structure_list.append(vacant_site)

            ref_specie = ref_site.specie

            # Get atomic number change and defect type
            add_property={
                "original_Z":ref_specie.Z,
                "new_Z": 0,

                "original_en": ref_specie.X,
                "new_en": 0.0,

                "original_ar": ref_specie.atomic_radius,
                "new_ar": 0.0,

                "original_row": ref_specie.row,
                "new_row": 0,

                "original_group": ref_specie.group,
                "new_group": 0,

                "original_max_os": max(ref_specie.common_oxidation_states),
                "new_max_os": 0,

                "original_ef": ref_specie.electron_affinity,
                "new_ef": 0.0,
                
                "vacancy_defect": 1.0,
                "substitution_defect": 0.0,
                "normal_site":0.0

            }
            defects_properties[vacant_site] = add_property

    # create a defects structure
    defective_struct = Structure.from_sites(defective_structure_list)

    # Add properties to defects structure
    for a_site in defective_struct.sites:
        if a_site in defects_properties.keys():
            a_site.properties.update(defects_properties[a_site])
        else:
            pass

    return defective_struct

# Testing
new_defective_structure = get_defects_structure(reference_structure, defective_structure)

In [5]:
def get_cloud(full_defective_structure):
    cloud_list = []

    all_sites = full_defective_structure.sites

    for site in all_sites:
        if site.properties["normal_site"] != 1.0:

            # Add site to list
            cloud_list.append(site)

    cloud_structure = Structure.from_sites(cloud_list)
        
    return cloud_structure

new_cloud = get_cloud(new_defective_structure)

## Create graphs for ML

In [6]:
# NODES

def get_nodes(defective_struct):
    defect_sites = defective_struct.sites

    nodes = []
    for site in defect_sites:
        coords = site.frac_coords.tolist()
        site_features = [
            site.properties["new_Z"]/94,
            site.properties["new_ar"],
            site.properties["new_ef"],
            site.properties["new_en"]/4,
            site.properties["new_group"]/18,
            site.properties["new_max_os"],
            site.properties["new_row"]/9,
            
            site.properties["original_Z"]/94,
            site.properties["original_ar"],
            site.properties["original_ef"],
            site.properties["original_en"]/4,
            site.properties["original_group"]/18,
            site.properties["original_max_os"],
            site.properties["original_row"]/9,

            site.properties["vacancy_defect"],
            site.properties["substitution_defect"],
            # site.properties["normal_site"],
        ]
        nodes.append(coords + site_features)

    return nodes

# defective_nodes = get_nodes(new_defective_structure)
cloud_nodes = get_nodes(new_cloud)

In [7]:
ref_lattice = reference_structure.lattice

def tensor_to_structure(tensor_structure, structure_lattice):
    all_sites = []

    all_properties = {}
    
    for row in tensor_structure:
        row = row.detach().cpu()
        points = {}
        
        frac_coords = row[:3].clamp(0.0, 1.0).numpy()

        # points["Zd"] = int((row[3] * 94).round().clamp(0, 94).item())
        Zd = int((row[3] * 94).round().clamp(0, 94).item())
        points["Zd"] = Zd
        points["ar_d"] = row[4].item()
        points["ef_d"] = row[5].item()
        points["en_d"] = int((row[6] * 4).round().clamp(0, 4).item())
        points["group_d"] = int((row[7] * 18).round().clamp(0, 18).item())
        points["max_os_d"] = row[8].item()
        points["row_d"] = int((row[9] * 9).round().clamp(0, 9).item())

        points["Zp"] = int((row[10] * 94).round().clamp(0, 94).item())
        points["ar_p"] = row[11].item()
        points["ef_p"] = row[12].item()
        points["en_p"] = int((row[13] * 4).round().clamp(0, 4).item())
        points["group_p"] = int((row[14] * 18).round().clamp(0, 18).item())
        points["max_os_p"] = row[15].item()
        points["row_p"] = int((row[16] * 9).round().clamp(0, 9).item())

        points["vac_defect"] = int(row[17].item())
        points["sub_defect"] = int(row[18].item())
        # points["norm_site"] = int(row[19].item())


        the_site = PeriodicSite(
            species= Element.from_Z(Zd).symbol if Zd > 0 else DummySpecie(),
            coords= frac_coords,
            coords_are_cartesian= False,
            lattice= structure_lattice
            )

        all_sites.append(the_site)
        all_properties[the_site] = points

    result_structure = Structure.from_sites(all_sites)

    # for a_site in result_structure.sites:
        # a_site.properties.update(all_properties[a_site])

    return result_structure

# defective_tensors = torch.tensor(defective_nodes, dtype=torch.float)
# defective_tensors_struct = tensor_to_structure(defective_tensors, ref_lattice)

cloud_tensors = torch.tensor(cloud_nodes, dtype=torch.float)
cloud_tensors_struct = tensor_to_structure(cloud_tensors, ref_lattice)

In [28]:
cloud_tensors.shape

torch.Size([3, 19])

In [ ]:
# For full defective structures
def get_edges_and_features(reference_structure):
    src, dst = [], []
    features = []
    coords = np.array([s.coords for s in reference_structure.sites])
    new_max_oss = np.array([s.properties["new_max_os"] for s in reference_structure])
    vacancys = np.array([s.properties["vacancy_defect"] for s in reference_structure])
    subs = np.array([s.properties["substitution_defect"] for s in reference_structure])
    norms = np.array([s.properties["normal_site"] for s in reference_structure])

    for i in range(len(reference_structure.sites)):
        for j in range(len(reference_structure.sites)):
            if i == j:
                continue
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < 5.0:
                src.append(i)
                dst.append(j)

                
                 # Electrostatic interaction
                q_i = new_max_oss[i]
                q_j = new_max_oss[j]

                if vacancys[i] == 1:
                    q_i = -q_i

                if vacancys[j] == 1:
                    q_j = -q_j

                charge_product = q_i * q_j
                screened_coulomb = (q_i * q_j) / (dist) if dist > 0 else 0.0

            

                # Defect interaction
                if (vacancys[i] == 1 and subs[j] == 1) or (subs[i] == 1 and vacancys[j] == 1) :
                    vac_sub = 1
                    vac_vac, vac_norm, sub_sub, sub_norm, norm_norm = 0,0,0,0,0

                elif (vacancys[i] == 1 and norms[j] == 1) or (norms[i]==1 and vacancys[j]==1):
                    vac_norm = 1
                    sub_sub, vac_vac, vac_sub, sub_norm, norm_norm = 0,0,0,0,0

                elif (subs[i] == 1 and norms[j] == 1) or (norms[i]==1 and subs[j]==1):
                    sub_norm = 1
                    sub_sub, vac_vac, vac_sub, vac_norm, norm_norm = 0,0,0,0,0

                elif subs[i] == 1 and subs[j] == 1:
                    sub_sub = 1
                    vac_vac, vac_norm, vac_sub, sub_norm, norm_norm = 0,0,0,0,0


                elif vacancys[i] == 1 and vacancys[j]== 1:
                    vac_vac = 1
                    sub_sub, vac_norm, vac_sub, sub_norm, norm_norm = 0,0,0,0,0

                elif norms[i] == 1 and norms[j]== 1:
                    norm_norm = 1
                    vac_vac, vac_norm, vac_sub, sub_norm, sub_sub = 0,0,0,0,0

                else:
                    vac_vac, vac_sub, vac_norm, sub_sub, sub_norm, norm_norm = 0,0,0,0,0,0

                features.append(
                    [
                        dist, charge_product, screened_coulomb, 
                        vac_vac, vac_sub, vac_norm, sub_sub, sub_norm, norm_norm 
                    ]
                )
    ed = [src, dst]
    
    return ed, features

defective_edges, defective_edge_features = get_edges_and_features(new_defective_structure)

In [8]:
# For cloud structures
def get_edges_and_features(reference_structure, defects_structure):
    a_lat = float(reference_structure.lattice.a)

    from_edge = []
    to_edge = []
    edges = []
    edge_features = []

    cart_coords_ds = defects_structure.cart_coords

    for i, site_i in enumerate(defects_structure):
        for j, site_j in enumerate(defects_structure):
            if j > i :
                # pass
            # else:
                from_edge.append(i)
                to_edge.append(j)

                cart_i = cart_coords_ds[i]
                cart_j = cart_coords_ds[j]
                r_vec = cart_j - cart_i
                r_ij = float(np.linalg.norm(r_vec))

                # if r_ij > 12.0 or r_ij<1e-6:
                    # continue

                dist_angstrom = r_ij
                dist_norm = site_i.distance(site_j)
                dist_lattice_units = r_ij/a_lat

                # Electrostatic interaction
                q_i = site_i.properties["new_max_os"]
                q_j = site_j.properties["new_max_os"]

                if site_i.properties["vacancy_defect"] == 1:
                    q_i = -q_i

                if site_j.properties["vacancy_defect"] == 1:
                    q_j = -q_j

                charge_product = q_i * q_j
                screened_coulomb = (q_i * q_j) / (r_ij) if r_ij > 0 else 0.0

                # Angular factor
                if r_ij > 0:
                    cos_theta = r_vec[2]/ r_ij
                    angular_factor = 1.0-3.0 * cos_theta ** 2
                else:
                    angular_factor = 0.0

                # Defect interaction
                if site_i.properties["vacancy_defect"] == 1 and site_j.properties["substitution_defect"] == 1:
                    vac_sub = 1
                    vac_vac = 0
                    sub_sub = 0

                elif site_i.properties["substitution_defect"] == 1 and site_j.properties["vacancy_defect"] == 1:
                    vac_sub = 1
                    vac_vac = 0
                    sub_sub = 0

                elif site_i.properties["substitution_defect"] == 1 and site_j.properties["substitution_defect"] == 1:
                    vac_sub = 0
                    vac_vac = 0
                    sub_sub = 1

                elif site_i.properties["vacancy_defect"] == 1 and site_j.properties["vacancy_defect"] == 1:
                    vac_sub = 0
                    vac_vac = 1
                    sub_sub = 0

                else:
                    vac_sub = 0
                    vac_vac = 0
                    sub_sub = 0

                edge_features.append(
                    [
                        dist_angstrom, dist_norm, dist_lattice_units,
                        charge_product, screened_coulomb,angular_factor, 
                        vac_vac, sub_sub, vac_sub
                    ]
                )

    edges.append(from_edge)
    edges.append(to_edge)

    return edges, edge_features

In [10]:
data_list = []

the_material = "high_GaSe"
elem = the_material.split("_")[1]

sample_data = comb_df[comb_df["dataset_material"] == the_material].reset_index(drop=True)
max_points = int(sample_data["defect_sites"].max())

all_materials = ["BN", "GaSe", "InSe", "MoS2", "P", "WSe2"]
one_hot = [1 if cat == elem else 0 for cat in all_materials]

ref_sample = Structure.from_file(f"{final_data}/ref_cifs/{the_material}.cif")



for index, row in sample_data.iterrows():
    id = row["_id"]
    bgv = row["band_gap_value"]
    defective_structure = Structure.from_file(f"{original_data}/{the_material}/cifs/{id}.cif")

    full_defective_structure = get_defects_structure(ref_sample, defective_structure)
    cloud_structure = get_cloud(full_defective_structure)

    # the_nodes = get_nodes(full_defective_structure)
    # the_edges, the_features = get_edges_and_features(full_defective_structure)
    the_nodes = get_nodes(cloud_structure)
    len_nodes = len(the_nodes)
    if len_nodes <max_points:
        # Pad with zeros
        padding = [[0.0]*len(the_nodes[0])]*(max_points - len_nodes)
        the_nodes.extend(padding)
        
    the_edges, the_features = get_edges_and_features(ref_sample, cloud_structure)

    # cloud_struct = get_cloud(full_defective_structure)

    condition = one_hot + [bgv]

    # global_features = graphy_gnn.get_globals(ref_sample, defective_structure, cloud_struct)

    # band_gap = torch.tensor([bgv], dtype=torch.float)
    
    data = Data(
        x=torch.tensor(the_nodes, dtype=torch.float),
        edge_index=torch.tensor(the_edges, dtype=torch.long),
        edge_attr=torch.tensor(the_features, dtype=torch.float),
        u=torch.tensor(condition, dtype=torch.float).unsqueeze(0),
        # u=torch.tensor(global_features, dtype=torch.float).unsqueeze(0),
        # y=torch.tensor(band_gap, dtype=torch.float).unsqueeze(0)
        )
    
    data_list.append(data)

In [11]:
sample_train, sample_val = train_test_split(data_list, test_size=0.25, random_state=42)

In [12]:
sample_train_loader = DataLoader(sample_train, batch_size=1, shuffle=True) 
sample_val_loader = DataLoader(sample_val, batch_size=1, shuffle=False)

for batch in sample_train_loader:
    print(batch)
    break

DataBatch(x=[18, 19], edge_index=[2, 91], edge_attr=[91, 9], u=[1, 7], batch=[18], ptr=[2])


## The model architecture

In [59]:
# Encoder2
class Encoder2(nn.Module):
    def __init__(self, node_dim, edge_dim, u_dim, hidden_dim, latent_dim):
        super().__init__()

        self.edge_nn = nn.Sequential(
            nn.Linear(edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, node_dim * hidden_dim)
        )

        self.conv0 = NNConv(node_dim, hidden_dim, self.edge_nn, aggr='mean')
        # self.conv1 = CGConv(hidden_dim, edge_dim, aggr='mean') # , batch_norm=True)

        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.global_embed = nn.Linear(u_dim, hidden_dim)

        # cloud_flat = n_max * point_dim
        
        self.fc_combine = nn.Linear(hidden_dim * 3, hidden_dim*2)
        self.linear2 = nn.Linear(hidden_dim*2, hidden_dim)
        self.linear3 = nn.Linear(hidden_dim, 64)

        self.fc_mu      = nn.Linear(64, latent_dim)
        self.fc_log_var = nn.Linear(64, latent_dim)

    def forward(self, data):
        x, edge_index, edge_attr, batch, u = (
            data.x,
            data.edge_index,
            data.edge_attr,
            data.batch,
            data.u,
        )
            
        x = F.relu(self.conv0(x, edge_index, edge_attr))
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x,batch)

        u = self.global_embed(u)

        x = torch.cat([x_max, x_mean], dim=1)
        x = torch.cat([x,u], dim=-1)

        hidden   = F.relu(self.fc_combine(x))
        hidden   = F.relu(self.linear2(hidden))
        hidden   = F.relu(self.linear3(hidden))
        
        mu = self.fc_mu(hidden) 
        log_var = self.fc_log_var(hidden) 
        
        return mu, log_var
    
# Decoder 2
class Decoder2(nn.Module):
    def __init__(self, hidden_dim, latent_dim, node_dim, u_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(latent_dim + u_dim, 64),
            nn.ReLU(),
            nn.Linear(64, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, 18 * node_dim),
        )

        # self.conv1 = CGConv(hidden_dim, hidden_dim)
        # self.conv2 = CGConv(hidden_dim, 144*node_dim)

    def forward(self, z, condition_vector):

        """if condition_vector.dim() == 1:
            condition_vector = condition_vector.unsqueeze(-1)"""

        inp = torch.cat([z, condition_vector], dim=-1)

        recon = self.net(inp).view((18, 19))

        coords = torch.sigmoid(recon[...,:3])  # Ensure coordinates are between 0 and 1
        props = recon[..., 3:17]
        logits = F.one_hot(torch.argmax(recon[...,17:], dim=-1), num_classes=2).float()
        
        fine_recon = torch.cat([coords, props, logits], dim=-1)

        return fine_recon 


# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim, edge_dim, u_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder2(node_dim, edge_dim, u_dim, hidden_dim, latent_dim)
        self.decoder = Decoder2(hidden_dim, latent_dim, node_dim, u_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.u)
        
        return recon, mu, log_var
        # return recon

    @torch.no_grad()
    def generate(self, reference_structure, target_band_gap, device = "cpu"):
        self.eval()

        the_lattice = reference_structure.lattice
        elem = reference_structure.reduced_formula

        all_materials = ["BN", "GaSe", "InSe", "MoS2", "P", "WSe2"]
        one_hot = [1 if cat == elem else 0 for cat in all_materials]

        condition = one_hot + [target_band_gap]


        z = torch.randn(1, 32, device=device)
        bg = torch.tensor(condition*1, dtype=torch.float).unsqueeze(0).to(device=device)
        
        defective_tensors = self.decoder(z, bg).view((18, 19))

        new_structure = tensor_to_structure(defective_tensors, the_lattice)

        return new_structure
        # return defective_tensors, the_lattice

In [50]:
NODE_DIM = sample_train[0].x.shape[1]
EDGE_DIM = sample_train[0].edge_attr.shape[1]
U_DIM = sample_train[0].u.shape[1]

HIDDEN_DIM = 128
LATENT_DIM = 32

In [51]:
# Test with first batch from loader
for batch in sample_train_loader:
    the_sample = batch
    break


In [60]:
the_model = DefectCVAE(NODE_DIM, EDGE_DIM, U_DIM, HIDDEN_DIM, LATENT_DIM)
recon, mu, log_var = the_model(the_sample)
recon = recon.view((18,NODE_DIM))
recon.shape

torch.Size([18, 19])

In [61]:
the_model.generate(ref_sample, 1.5)

Structure Summary
Lattice
    abc : 22.91440026 22.91440026 20.0
 angles : 90.0 90.0 120.00001238000002
 volume : 9094.473524893241
      A : np.float64(22.91440026) np.float64(0.0) np.float64(1.403102346639514e-15)
      B : np.float64(-11.457204417825597) np.float64(19.844450262066868) np.float64(1.403102346639514e-15)
      C : np.float64(0.0) np.float64(0.0) np.float64(20.0)
    pbc : True True True
PeriodicSite: C (5.817, 9.638, 9.69) [0.4967, 0.4857, 0.4845]
PeriodicSite: N (5.732, 10.43, 10.31) [0.5129, 0.5255, 0.5154]
PeriodicSite: X0+ (5.702, 9.982, 9.91) [0.5003, 0.503, 0.4955]
PeriodicSite: Ne (6.123, 9.833, 9.892) [0.515, 0.4955, 0.4946]
PeriodicSite: X0+ (5.957, 10.08, 10.01) [0.514, 0.5079, 0.5007]
PeriodicSite: O (6.236, 9.36, 9.983) [0.508, 0.4717, 0.4992]
PeriodicSite: X0+ (4.584, 10.18, 9.825) [0.4566, 0.5131, 0.4912]
PeriodicSite: X0+ (5.415, 9.763, 10.22) [0.4823, 0.492, 0.5108]
PeriodicSite: X0+ (5.598, 10.05, 10.1) [0.4975, 0.5064, 0.5052]
PeriodicSite: He (5.696,

### Loss Function

In [64]:
# Loss function for cvae
def cvae_loss(recon, target, mu, log_var, beta): # =1e-3):

    
    # coords
    recon_coords = recon[..., :3]
    target_coords = target[..., :3]
    coords_loss = F.mse_loss(recon_coords, target_coords)

    # Props
    recon_props = recon[..., 3:17]
    target_props = target[..., 3:17]
    props_loss = F.mse_loss(recon_props, target_props)

    # sites
    recon_sites = recon[..., 17:]
    target_sites = target[..., 17:]
    sites_loss = F.cross_entropy(recon_sites, target_sites)


    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

    total = coords_loss + props_loss + sites_loss + beta * kl_loss
    return total, coords_loss, props_loss, sites_loss, kl_loss

cvae_loss(recon, the_sample.x, mu, log_var, 1e-3)

(tensor(3.4777, grad_fn=<AddBackward0>),
 tensor(0.0642, grad_fn=<MseLossBackward0>),
 tensor(2.6002, grad_fn=<MseLossBackward0>),
 tensor(0.8133, grad_fn=<DivBackward1>),
 tensor(0.0158, grad_fn=<MulBackward0>))

## Training and Testing

In [66]:
# Loaders
train_loader = sample_train_loader # DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = sample_val_loader # DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Parameters for training model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DefectCVAE(NODE_DIM, EDGE_DIM, U_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
EPOCHS = 40
beta = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.001)
losses_dict = {"train_loss": [], "val_loss": [], "kl": [], "mse": []}

# Validate function
def validate(model, loader, beta):
    # Model in evaluation mode
    model.eval()
    t_loss, tc_loss, tp_loss, ts_loss, kl = 0.0, 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():
        for batch in loader:
            # batch = {k: v.to(device) for k, v in batch.items()}
            recon, mu, log_var = model(batch.to(device))
            recon = recon.view((18,19))

            t, c_loss, p_loss, s_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, beta)

            t_loss +=t.item() * batch.num_graphs
            tc_loss += c_loss.item() * batch.num_graphs
            tp_loss += p_loss.item() * batch.num_graphs
            ts_loss += s_loss.item() * batch.num_graphs
            kl += kl_loss.item() * batch.num_graphs

        n = len(loader.dataset)
        totals = {"the_loss": t_loss/n, "coordinates": tc_loss/n, "props": tp_loss/n, "sites": ts_loss/n ,"kl": kl/n}
        return totals

# Training loop
for epoch in range(1, EPOCHS+1):
    # Put model in training mode at the start of each epoch
    model.train()
    t_loss, tc_loss, tp_loss, ts_loss, kl = 0.0, 0.0, 0.0, 0.0, 0.0
    current_beta = beta * min(1.0, epoch / EPOCHS)

    for batch in train_loader:
        optimizer.zero_grad()

        # Output of the model
        recon, mu, log_var = model(batch.to(device))
        recon = recon.view((18, 19))

        
        t, c_loss, p_loss, s_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, current_beta)

        t.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        t_loss +=t.item() * batch.num_graphs
        tc_loss += c_loss.item() * batch.num_graphs
        tp_loss += p_loss.item() * batch.num_graphs
        ts_loss += s_loss.item() * batch.num_graphs
        kl += kl_loss.item() * batch.num_graphs

    n = len(train_loader)
    # t_losses = {"the_loss": t_loss/n, "mse": t_mse/n, "kl": t_kl/n}
    t_losses = {"the_loss": t_loss/n, "coordinates": tc_loss/n, "props": tp_loss/n, "sites": ts_loss/n ,"kl": kl/n}

    # Validate at the end of each epoch
    val_losses = validate(model, val_loader, current_beta)
    scheduler.step()

    print(f"Epoch {epoch:4d}\ntrain: {t_losses['the_loss']:.4f} | {t_losses['coordinates']:.4f} | {t_losses['props']:.4f} | {t_losses['sites']:.4f} | {t_losses['kl']:.4f}\nval: {val_losses['the_loss']:.4f} | {val_losses['coordinates']:.4f} | {val_losses['props']:.4f} | {val_losses['sites']:.4f} | {val_losses['kl']:.4f}\n")

Epoch    1
train: 1.3951 | 0.0823 | 0.8366 | 0.4762 | 2.1270
val: 1.1681 | 0.0601 | 0.6356 | 0.4724 | 1.2721

Epoch    2
train: 1.1798 | 0.0528 | 0.6546 | 0.4723 | 1.2166
val: 1.0346 | 0.0450 | 0.5344 | 0.4550 | 1.8253

Epoch    3
train: 1.0080 | 0.0411 | 0.4918 | 0.4750 | 1.8236
val: 0.9838 | 0.0382 | 0.4895 | 0.4559 | 3.0371

Epoch    4
train: 1.0130 | 0.0399 | 0.4969 | 0.4760 | 2.0536
val: 0.8805 | 0.0350 | 0.3863 | 0.4590 | 2.3404

Epoch    5
train: 0.8913 | 0.0346 | 0.3824 | 0.4739 | 2.7973
val: 0.8413 | 0.0331 | 0.3442 | 0.4635 | 4.0246

Epoch    6
train: 0.9246 | 0.0361 | 0.4127 | 0.4754 | 2.4670
val: 0.8440 | 0.0325 | 0.3492 | 0.4617 | 3.8080

Epoch    7
train: 0.8641 | 0.0330 | 0.3540 | 0.4766 | 3.0625
val: 0.8465 | 0.0326 | 0.3547 | 0.4586 | 3.3482

Epoch    8
train: 0.9333 | 0.0362 | 0.4237 | 0.4729 | 2.6160
val: 0.8515 | 0.0321 | 0.3522 | 0.4666 | 3.2177

Epoch    9
train: 0.8910 | 0.0343 | 0.3810 | 0.4750 | 3.1809
val: 0.8303 | 0.0322 | 0.3351 | 0.4621 | 3.8668

Epoch   10

## Inverse Design

In [67]:
def inverse_design(model, pristine, target_band_gap, device):
    model.eval()
    model = model.to(device)

    # Generate defective structure
    gen_def_structure = model.generate(pristine, target_band_gap)

   
    return gen_def_structure

origin = Structure.from_file(f"{final_data}/ref_cifs/high_GaSe.cif")
first_struct = inverse_design(model, origin, 1.3, "cpu")

In [68]:
print(origin)

Full Formula (Ga72 Se72)
Reduced Formula: GaSe
abc   :  22.914400  22.914400  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (144)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Ga    0.111111  0.055556  0.383273
  1  Ga    0.111111  0.222222  0.383273
  2  Ga    0.111111  0.388889  0.383273
  3  Ga    0.111111  0.555556  0.383273
  4  Ga    0.111111  0.722222  0.383273
  5  Ga    0.111111  0.888889  0.383273
  6  Ga    0.277778  0.055556  0.383273
  7  Ga    0.277778  0.222222  0.383273
  8  Ga    0.277778  0.388889  0.383273
  9  Ga    0.277778  0.555556  0.383273
 10  Ga    0.277778  0.722222  0.383273
 11  Ga    0.277778  0.888889  0.383273
 12  Ga    0.444444  0.055556  0.383273
 13  Ga    0.444444  0.222222  0.383273
 14  Ga    0.444444  0.388889  0.383273
 15  Ga    0.444444  0.555556  0.383273
 16  Ga    0.444444  0.722222  0.383273
 17  Ga    0.444444  0.888889  0.383273
 18  Ga    0.61111

In [69]:
print(first_struct)

Full Formula (X6 K1 Na2 Li2 V1 Cr1 Fe1 Cl2 Ar1 He1)
Reduced Formula: X6KNa2Li2VCrFeCl2ArHe
abc   :  22.914400  22.914400  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (18)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  K     0.303369  0.415502  0.398492
  1  Cl    0.449973  0.453998  0.38383
  2  Cl    0.44845   0.402469  0.391213
  3  Fe    0.451263  0.427455  0.381593
  4  Cr    0.600845  0.48885   0.391799
  5  Ar    0.604227  0.450686  0.423543
  6  V     0.612987  0.461539  0.454698
  7  Li    0.148855  0.151272  0.14664
  8  X0+   0.077479  0.076881  0.123258
  9  Na    0.123276  0.085447  0.100688
 10  X0+   0.053404  0.088977  0.070574
 11  He    0.058427  0.076351  0.072696
 12  X0+   0.07192   0.0755    0.055681
 13  Na    0.07449   0.082825  0.049215
 14  X0+   0.038697  0.030946  0.035206
 15  X0+   0.033589  0.043198  0.025711
 16  X0+   0.044491  0.036614  0.030962
 17  Li    0.0514

/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/composition.py:1366: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
/home/amutua/inverse/lib/python3.10/site-packages/p